In [1]:
import requests
import time
from datetime import datetime
import pandas as pd

# Configure headers - Reddit requires a User-Agent
HEADERS = {
    'User-Agent': 'NASCAR-Visibility-Analysis/1.0 (Educational Project)'
}

# Test connection
url = "https://www.reddit.com/r/NASCAR/about.json"
response = requests.get(url, headers=HEADERS)

if response.status_code == 200:
    data = response.json()['data']
    print("Reddit JSON API connected!")
    print(f"Subreddit: r/{data['display_name']}")
    print(f"Subscribers: {data['subscribers']:,}")
else:
    print(f"Error: Status code {response.status_code}")

Reddit JSON API connected!
Subreddit: r/NASCAR
Subscribers: 1,486,726


### Search for Sponsor Mentions

In [2]:
# Search r/NASCAR for mentions of a sponsor
# Start with a major sponsor for a good test (FedEx is frequently discussed)
search_query = "FedEx"

print(f"Searching r/NASCAR for '{search_query}'...")
print("-" * 50)

# Pause after first request (rate limiting)
time.sleep(2)

# Search using JSON API
url = "https://www.reddit.com/r/NASCAR/search.json"
params = {
    'q': search_query,
    'restrict_sr': 1,  # Restrict to r/NASCAR
    'sort': 'new',
    't': 'year',  # Past year
    'limit': 25
}

response = requests.get(url, headers=HEADERS, params=params)

posts_data = []
if response.status_code == 200:
    posts = response.json()['data']['children']

    for post in posts:
        p = post['data']
        post_info = {
            'title': p['title'],
            'score': p['score'],
            'num_comments': p['num_comments'],
            'created_utc': datetime.fromtimestamp(p['created_utc']),
            'url': f"https://reddit.com{p['permalink']}",
            'selftext_preview': p.get('selftext', '')[:100]
        }
        posts_data.append(post_info)

    print(f"Found {len(posts_data)} posts mentioning '{search_query}'")
else:
    print(f"Error: Status code {response.status_code}")

Searching r/NASCAR for 'FedEx'...
--------------------------------------------------
Found 12 posts mentioning 'FedEx'


### Analyze Results

In [3]:
# Convert to DataFrame
df = pd.DataFrame(posts_data)

# Display results
print("\nTop posts by engagement:")
print(df.sort_values('score', ascending=False)[['title', 'score', 'num_comments', 'created_utc']].head(10))

# Basic statistics
print(f"\nStatistics for '{search_query}' mentions:")
print(f"  Total posts found: {len(df)}")
print(f"  Average score: {df['score'].mean():.1f}")
print(f"  Average comments: {df['num_comments'].mean():.1f}")
print(f"  Date range: {df['created_utc'].min().date()} to {df['created_utc'].max().date()}")


Top posts by engagement:
                                                title  score  num_comments  \
2   What if NASCAR required drivers to keep the sa...    597           200   
4                    I think they are a little behind    429            31   
3   FOX Sports has released their 2026 Daytona 500...    385           121   
11            When did Denny Hamlin become a villain?    142           220   
6                         Battered Denny Fan Syndrome    100            24   
5   23XI Racing and Front Row Motorsports v. NASCA...     92           907   
8   Beech Ridge Motor Speedway torn down to make w...     65            14   
9                                       Picture help!     54            11   
10                    2nd Year Nascar Fan Perspective     26            11   
7   Predictions for Kaulig trucks numbers/driver l...      1            50   

           created_utc  
2  2026-04-03 16:43:22  
4  2025-12-26 13:05:44  
3  2026-01-12 21:27:28  
11 2025-06-08 2

In [ ]:
import os

# Save to the project's data/raw folder (../ goes up from code/ to project root)
os.makedirs('../data/raw', exist_ok=True)

df.to_csv('../data/raw/test_reddit_mentions.csv', index=False)
print(f"\nTest data saved to data/raw/test_reddit_mentions.csv")